In [4]:
%pip install pandas psycopg2-binary python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values
import os
from dotenv import load_dotenv

load_dotenv()
conn = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    dbname=os.getenv("DB_NAME"),
)
cursor = conn.cursor()

df = pd.read_csv("data/NYC311_EDA.csv")
print(df.shape)

(99998, 18)


In [7]:
df["created_date"] = pd.to_datetime(df["created_date"])
df["closed_date"]  = pd.to_datetime(df["closed_date"])

for c in ["bad_dates", "bad_location", "is_closed", "is_outlier"]:
    df[c] = df[c].astype(bool)

df = df.astype(object).where(pd.notna(df), None)
print("Ready")

Ready


In [8]:
cols = list(df.columns)
sql = f"""INSERT INTO service_requests ({', '.join(cols)}) VALUES %s
          ON CONFLICT (unique_key) DO NOTHING"""

rows = [tuple(r) for r in df.values.tolist()]
batch = 5000
for i in range(0, len(rows), batch):
    execute_values(cursor, sql, rows[i:i + batch])
    conn.commit()
    print("Inserted up to row", i + batch)

Inserted up to row 5000
Inserted up to row 10000
Inserted up to row 15000
Inserted up to row 20000
Inserted up to row 25000
Inserted up to row 30000
Inserted up to row 35000
Inserted up to row 40000
Inserted up to row 45000
Inserted up to row 50000
Inserted up to row 55000
Inserted up to row 60000
Inserted up to row 65000
Inserted up to row 70000
Inserted up to row 75000
Inserted up to row 80000
Inserted up to row 85000
Inserted up to row 90000
Inserted up to row 95000
Inserted up to row 100000


In [9]:
cursor.execute("SELECT COUNT(*) FROM service_requests")
db_rows = cursor.fetchone()[0]
print("CSV rows:       ", len(df))
print("PostgreSQL rows:", db_rows)
print("Match:", len(df) == db_rows)

CSV rows:        99998
PostgreSQL rows: 99998
Match: True
